# Détection de profils atypiques sur Twitter — Récap

## 1. Contexte et objectif

Comparaison de **deux approches de classification** pour détecter des profils **atypiques** sur Twitter :

| Approche | Méthodes comparées | Retenue | Notebook |
|----------|-------------------|---------|----------|
| Non supervisée | K-Means vs Isolation Forest | Isolation Forest | `Groupe3_profils_atypiques_non_Sup.ipynb` |
| Supervisée | SVM vs XGBoost | XGBoost | `Groupe3_profils_atypiques_Sup.ipynb` |

**Jeu de données :** `users_labeled_manual.csv` (643 124 profils, ~17 % atypiques)

Source : ~1,16 M tweets agrégés par utilisateur (MongoDB).

## 2. Méthodologie commune

**Pipeline :** features -> `StandardScaler` -> **ACP** -> modèles

| Contexte | Features | Composantes ACP |
|----------|----------|-----------------|
| Non supervisé (16 feat.) | Seuil 75 % variance | **7 (~79 %)** |
| Supervisé (8 feat.) | Saturation 100 % | **5** |

> **k=7** K-Means = nombre de **clusters** (coude inertie), indépendant de l'ACP.

| Étape | Non supervisé | Supervisé |
|-------|---------------|-----------|
| Features | 16 MongoDB | 8 hors règles Excel |
| Normalisation | StandardScaler | StandardScaler (fit train) |
| ACP | 7 composantes | 5 composantes |
| Modèles | K-Means, Isolation Forest | SVM, XGBoost |

**8 features supervisées (hors règles) :** `followers_count`, `friends_count`, `avg_tweet_length`, `avg_hashtags`, `avg_favorites`, `avg_retweet_count`, `verified`, `default_profile_image`.

## 3. Approche non supervisée — K-Means vs Isolation Forest

**Fichier requis :** `users_labeled_manual.csv` — **643 124 lignes** (jeu partiel → K-Means peut retourner 0 atypique).

### 3.1 K-Means (MiniBatchKMeans, k = 7 clusters)

- Clustering sur l'espace ACP (**7 composantes**, ~79 % variance)
- Profils atypiques = clusters minoritaires (**effectif < 1 %** du jeu)
- Résultat : **3 498 profils (0,54 %)** — cluster 4 uniquement

### 3.2 Isolation Forest

- `contamination = 'auto'` (seuil déterminé par les scores d'anomalie)
- Résultat : **42 987 profils (6,68 %)**

### 3.3 Concordance K-Means ∩ Isolation Forest

| Catégorie | Profils | % du jeu |
|-----------|---------|----------|
| K-Means seulement | **0** | 0,00 % |
| **Les deux méthodes** | **3 498** | **0,54 %** |
| IsoForest seulement | **39 489** | 6,14 % |

> Tous les atypiques K-Means sont aussi détectés par IF (**K-Means ⊆ IF**). Le consensus (3 498) constitue un noyau dur robuste.

### 3.4 Méthode retenue : Isolation Forest

Conçue pour la détection d'anomalies, plus sensible que K-Means sur 643 000 profils.

## 4. Approche supervisée — SVM vs XGBoost

Split train/test 80/20 stratifié. Pipeline : StandardScaler -> **ACP (5 comp.)** -> modèle.

### Limite méthodologique

Les labels suivent 4 règles Excel. Avec **16 features + ACP 7 comp.**, F1 ~ 0,970 (circularité).
Évaluation retenue : **8 features hors règles + ACP 5 comp.**

### 4.3 Comparaison (ACP 5 comp. + 8 features hors règles)

| Métrique | SVM | XGBoost |
|----------|-----|---------|
| Accuracy | 55,9 % | **66,7 %** |
| F1 (Atypique) | 0,361 | **0,443** |
| Recall | 0,737 | **0,783** |
| Precision | 0,239 | **0,309** |
| ROC-AUC | 0,670 | **0,794** |

### Modèle retenu : XGBoost

Meilleur F1 et ROC-AUC hors règles de labellisation.

## 5. Comparaison des deux approches retenues

| | Isolation Forest | XGBoost |
|--|------------------|---------|
| Labels requis | Non | Oui (Excel) |
| Features ML | 16 | 8 (hors règles) |
| ACP | 7 composantes (~79 %) | 5 composantes |
| Détection / F1 vs labels | 42 987 (6,68 %) · consensus KM∩IF : 3 498 | F1 = 0,443 · ROC-AUC = 0,794 |
| Forces | Exploration sans annotation | Signal indirect avec labels |
| Faiblesses | Seuil auto (non contrôlé) | Labels corrélés aux règles Excel |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Métriques validées — jeu complet users_labeled_manual.csv (643 124 profils)
N_TOTAL = 643_124

RESULTS_NS = {
    "K-Means\nseulement": {"count": 0, "pct": 0.00},
    "Les DEUX\nméthodes": {"count": 3_498, "pct": 0.54},
    "IsoForest\nseulement": {"count": 39_489, "pct": 6.14},
}
RESULTS_SUP = {"SVM": 0.361, "XGBoost": 0.443}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Non supervisé ---
categories_ns = list(RESULTS_NS.keys())
values_ns = [v["count"] for v in RESULTS_NS.values()]
colors_ns = ["#2196F3", "#9C27B0", "#F44336"]
bars_ns = axes[0].bar(categories_ns, values_ns, color=colors_ns, edgecolor="black", width=0.55)
axes[0].set_title(
    "Non supervisé — Concordance K-Means vs Isolation Forest\n(Isolation Forest retenu)",
    fontweight="bold",
)
axes[0].set_ylabel("Nombre de profils")
axes[0].grid(axis="y", alpha=0.4)
ymax_ns = max(values_ns) if max(values_ns) > 0 else 1
offset_ns = max(500, ymax_ns * 0.03)
for bar, key in zip(bars_ns, RESULTS_NS):
    val = RESULTS_NS[key]["count"]
    pct = RESULTS_NS[key]["pct"]
    pct_str = f"{pct:.3f} %" if pct < 1 else f"{pct:.2f} %"
    y_text = max(bar.get_height(), ymax_ns * 0.02) + offset_ns
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        y_text,
        f"{val:,}\n({pct_str})",
        ha="center",
        fontweight="bold",
        fontsize=9,
    )

# --- Supervisé ---
models = list(RESULTS_SUP.keys())
f1_scores = list(RESULTS_SUP.values())
bars_sup = axes[1].bar(models, f1_scores, color=["#2196F3", "#4CAF50"], edgecolor="black", width=0.5)
axes[1].set_title(
    "Supervisé — F1 (ACP 5 comp., 8 feat. hors règles)\n(XGBoost retenu)",
    fontweight="bold",
)
axes[1].set_ylabel("F1-score")
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis="y", alpha=0.4)
for bar, val in zip(bars_sup, f1_scores):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.3f}",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

In [ ]:
# Tableau récapitulatif final
N_TOTAL = 643_124
recap = pd.DataFrame({
    "Approche": ["Non supervisée", "Supervisée"],
    "Méthodes comparées": ["K-Means vs Isolation Forest", "SVM vs XGBoost"],
    "Méthode retenue": ["Isolation Forest", "XGBoost"],
    "Détection / Performance": [
        "IF : 42 987 (6,68 %) · Consensus KM∩IF : 3 498 (0,54 %)",
        "F1 = 0,443 · ROC-AUC = 0,794 (ACP 5 comp., 8 feat.)",
    ],
    "Notebook": [
        "Groupe3_profils_atypiques_non_Sup.ipynb",
        "Groupe3_profils_atypiques_Sup.ipynb",
    ],
})

print("=" * 70)
print(" SYNTHÈSE FINALE — MÉTHODES RETENUES")
print(f" Jeu de données : {N_TOTAL:,} profils · ~16,9 % atypiques (labels manuels)")
print("=" * 70)
display(recap)

## 6. Conclusion

Quatre méthodes comparées en deux approches complémentaires :

1. **Isolation Forest** (non supervisée) — **42 987** profils atypiques (~6,68 %, `contamination='auto'`). ACP **7 composantes** (16 features). Concordance avec K-Means : **3 498** profils (noyau dur, 0 % hors IF).

2. **XGBoost** (supervisée) — F1 = **0,443**, ROC-AUC = **0,794**. ACP **5 composantes** (8 features hors règles Excel).

**Recommandation :** Isolation Forest en exploration (sans labels) ; XGBoost une fois les labels définis, sans réutiliser les variables des règles Excel en entrée.

**Livrables :** notebooks détaillés · `docs/RAPPORT_PROJET.md` · portail `demo/app.py`

---

*Projet IF29 — Groupe 3*